# Módulo 3 — Ajuste fino de *transformers* pré-treinados para NLP
**Objetivo de aprendizagem:** implementar um sistema pré-treinado para a geração de respostas automáticas.

**Caso:** o chat da LojaAurora precisa responder perguntas de clientes **com as políticas da loja**
(prazo de troca, frete, cupons...). Vamos comparar três caminhos:

1. **Só prompt:** um LLM pré-treinado, sem ajuste.
2. **Fine-tuning com LoRA:** ensinar as políticas ao modelo.
3. **RAG:** buscar a política certa e entregar como contexto, sem treinar nada.

| Tempo | Etapa |
|---|---|
| 5 min | Panorama: BERT, GPT, T5, Vertex AI/Gemini e o Hugging Face Hub |
| 5 min | Baseline: LLM pré-treinado sem ajuste |
| 10 min | Fine-tuning com LoRA |
| 5 min | RAG e comparação final |

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
MODO_RAPIDO = True   # True = subamostra para caber nos tempos da aula
SMOKE = os.environ.get("AULA_SMOKE") == "1"   # só para teste automatizado em CPU

if IN_COLAB:
    !pip install -q -U transformers peft accelerate sentence-transformers

import torch
import transformers
transformers.logging.set_verbosity_error()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| MODO_RAPIDO:", MODO_RAPIDO)
if DEVICE == "cpu" and not SMOKE:
    print("⚠️  Sem GPU. No Colab: Ambiente de execução → Alterar tipo → GPU T4.")

## 1. Panorama
| Família | Arquitetura | Quem | Onde usar |
|---|---|---|---|
| **BERT** | encoder | Google (2018) | classificação, NER, embeddings para busca |
| **GPT** | decoder | OpenAI | geração, chat |
| **T5 / Flan-T5** | encoder-decoder | Google | texto → texto (resumo, tradução) |
| **PaLM → Gemini** | decoder | Google | o PaLM 2 foi **descontinuado**, e o sucessor é o Gemini |
| **Vertex AI** | plataforma | Google Cloud | hospeda Gemini e modelos abertos, com fine-tuning gerenciado e MLOps |

**Hugging Face Hub:** mais de 1 milhão de modelos abertos. Ao escolher um modelo, olhe o *model card*:
licença (uso comercial?), tamanho (cabe na minha GPU?), idiomas, dados de treino, métricas e limitações.

In [ ]:
from huggingface_hub import HfApi

print("Modelos de geração de texto mais baixados:")
for m in HfApi().list_models(pipeline_tag="text-generation", sort="downloads", limit=8):
    print(f"  {m.id:55} {m.downloads:>12,}")

## 2. A base de conhecimento: FAQ da LojaAurora
São 25 políticas (empresa fictícia, com valores inventados de propósito) e 5 formas diferentes de perguntar cada uma.
Usamos **4 perguntas para treinar** e **guardamos 1 para testar**. O teste usa perguntas que o modelo nunca viu.

In [ ]:
FAQ = [{'intencao': 'prazo de troca',
  'chave': '45 dias',
  'resposta': 'Você tem 45 dias corridos após o recebimento para trocar qualquer produto, sem custo.',
  'perguntas': ['Qual o prazo para trocar um produto?',
                'Quantos dias eu tenho para fazer a troca?',
                'Comprei e não gostei, até quando posso trocar?',
                'Ainda dá tempo de trocar um item que recebi mês passado?',
                'Até quantos dias depois da entrega aceitam troca?']},
 {'intencao': 'frete grátis',
  'chave': 'R$ 179',
  'resposta': 'O frete é grátis para compras acima de R$ 179 para todo o Brasil.',
  'perguntas': ['Vocês têm frete grátis?',
                'A partir de quanto o frete sai de graça?',
                'Qual o valor mínimo para não pagar entrega?',
                'Tem entrega gratuita?',
                'Quanto preciso gastar para ganhar frete grátis?']},
 {'intencao': 'rastreio',
  'chave': 'AUR',
  'resposta': 'Seu código de rastreio começa com AUR e chega por e-mail; acompanhe em aurora.com.br/rastreio.',
  'perguntas': ['Como rastreio meu pedido?',
                'Onde vejo onde está minha encomenda?',
                'Qual é o código de rastreamento?',
                'Como acompanho a entrega?',
                'Não sei onde está meu pacote, como consulto?']},
 {'intencao': 'horário de atendimento',
  'chave': '8h às 22h',
  'resposta': 'Nosso atendimento humano funciona todos os dias, das 8h às 22h, pelo chat e pelo WhatsApp.',
  'perguntas': ['Qual o horário do atendimento?',
                'Até que horas vocês atendem?',
                'O SAC funciona no domingo?',
                'Que horas posso falar com um atendente?',
                'Vocês atendem de madrugada?']},
 {'intencao': 'parcelamento',
  'chave': '12x',
  'resposta': 'Parcelamos em até 12x sem juros no cartão para compras acima de R$ 300.',
  'perguntas': ['Posso parcelar?',
                'Em quantas vezes sem juros?',
                'Qual o parcelamento máximo no cartão?',
                'Dá para dividir a compra no crédito?',
                'Vocês parcelam sem juros?']},
 {'intencao': 'cashback pix',
  'chave': '3%',
  'resposta': 'Pagando com Pix você recebe 3% de cashback na sua carteira Aurora em até 2 dias.',
  'perguntas': ['Tem desconto no Pix?',
                'Vale a pena pagar com Pix?',
                'Qual a vantagem de pagar no Pix?',
                'Pix dá cashback?',
                'Quanto volta se eu pagar via Pix?']},
 {'intencao': 'prazo de reembolso',
  'chave': '5 dias úteis',
  'resposta': 'O reembolso é feito em até 5 dias úteis após recebermos o produto devolvido.',
  'perguntas': ['Quando recebo meu dinheiro de volta?',
                'Qual o prazo do reembolso?',
                'Devolvi o produto, quando cai o estorno?',
                'Em quanto tempo vocês devolvem o dinheiro?',
                'Quanto tempo demora a restituição?']},
 {'intencao': 'produto com defeito',
  'chave': 'coleta',
  'resposta': 'Para produto com defeito, abra um chamado em Meus Pedidos e agendamos a coleta gratuita na sua casa.',
  'perguntas': ['Meu produto veio com defeito, o que faço?',
                'Chegou quebrado, como resolvo?',
                'O aparelho parou de funcionar, como devolvo?',
                'Recebi um item danificado.',
                'O produto não liga, e agora?']},
 {'intencao': 'cancelamento',
  'chave': '2 horas',
  'resposta': 'Você pode cancelar sozinho em Meus Pedidos em até 2 horas após a compra; depois disso, fale com o '
              'atendimento.',
  'perguntas': ['Como cancelo um pedido?',
                'Quero cancelar minha compra.',
                'Ainda dá para cancelar o pedido que fiz agora?',
                'Me arrependi da compra, como desfaço?',
                'Qual o prazo para cancelar sozinho?']},
 {'intencao': 'entrega expressa',
  'chave': 'Aurora Já',
  'resposta': 'Com o Aurora Já, pedidos feitos até 13h nas capitais chegam no mesmo dia.',
  'perguntas': ['Tem entrega no mesmo dia?',
                'Como recebo mais rápido?',
                'Vocês têm entrega expressa?',
                'Se eu comprar de manhã chega hoje?',
                'Qual a entrega mais rápida de vocês?']},
 {'intencao': 'retirada em loja',
  'chave': 'armários',
  'resposta': 'Você pode retirar sem frete nos armários Aurora em estações de metrô, com o QR code enviado por e-mail.',
  'perguntas': ['Posso retirar o pedido em algum lugar?',
                'Tem retirada sem pagar frete?',
                'Vocês têm ponto de retirada?',
                'Dá para buscar a compra pessoalmente?',
                'Como funciona a retirada?']},
 {'intencao': 'programa de fidelidade',
  'chave': 'Aurora+',
  'resposta': 'No programa Aurora+ você acumula 1 ponto por real gasto e troca 1.000 pontos por R$ 25 em compras.',
  'perguntas': ['Vocês têm programa de pontos?',
                'Como funciona a fidelidade?',
                'Quanto valem meus pontos?',
                'Ganho alguma coisa comprando sempre?',
                'Como troco meus pontos?']},
 {'intencao': 'garantia estendida',
  'chave': '24 meses',
  'resposta': 'A Garantia Aurora estende a cobertura de eletrônicos por mais 24 meses e pode ser contratada até 30 '
              'dias após a compra.',
  'perguntas': ['Vocês vendem garantia estendida?',
                'Posso aumentar a garantia do celular?',
                'Quanto tempo de garantia extra posso comprar?',
                'Ainda dá para contratar garantia depois da compra?',
                'Como funciona a garantia estendida?']},
 {'intencao': 'nota fiscal',
  'chave': 'Meus Pedidos',
  'resposta': 'A nota fiscal fica disponível em PDF na página Meus Pedidos assim que o pedido é enviado.',
  'perguntas': ['Onde pego a nota fiscal?',
                'Preciso da NF da compra.',
                'Como emito a nota fiscal do pedido?',
                'Vocês mandam nota fiscal?',
                'Onde baixo a DANFE?']},
 {'intencao': 'cupom de primeira compra',
  'chave': 'BEMVINDO15',
  'resposta': 'Na primeira compra use o cupom BEMVINDO15 para ganhar 15% de desconto, limitado a R$ 60.',
  'perguntas': ['Tem cupom para primeira compra?',
                'Sou cliente novo, ganho desconto?',
                'Qual o cupom de boas-vindas?',
                'Existe código promocional para quem nunca comprou?',
                'Tem desconto no primeiro pedido?']},
 {'intencao': 'boleto',
  'chave': '3 dias úteis',
  'resposta': 'O boleto vence em 3 dias úteis e o pedido só é separado após a compensação do pagamento.',
  'perguntas': ['Qual o vencimento do boleto?',
                'Posso pagar com boleto?',
                'Quanto tempo tenho para pagar o boleto?',
                'Paguei o boleto, quando sai o pedido?',
                'O boleto vence quando?']},
 {'intencao': 'alterar endereço',
  'chave': 'antes do envio',
  'resposta': 'O endereço de entrega pode ser alterado pelo chat apenas antes do envio do pedido.',
  'perguntas': ['Posso mudar o endereço de entrega?',
                'Errei o endereço, e agora?',
                'Como troco o endereço do pedido?',
                'Me mudei, dá para alterar o destino?',
                'Coloquei o CEP errado, consigo corrigir?']},
 {'intencao': 'entrega internacional',
  'chave': 'não fazemos',
  'resposta': 'No momento não fazemos entregas internacionais; enviamos apenas para endereços no Brasil.',
  'perguntas': ['Vocês entregam fora do Brasil?',
                'Enviam para Portugal?',
                'Fazem entrega internacional?',
                'Moro nos EUA, posso comprar?',
                'Dá para mandar para outro país?']},
 {'intencao': 'produto esgotado',
  'chave': 'Avise-me',
  'resposta': 'Clique em Avise-me na página do produto esgotado e enviamos um e-mail quando ele voltar ao estoque.',
  'perguntas': ['O produto está esgotado, vai voltar?',
                'Como sei quando chegar no estoque?',
                'Item indisponível, o que faço?',
                'Vocês avisam quando repõem?',
                'Quando volta o produto que acabou?']},
 {'intencao': 'vale-presente',
  'chave': '12 meses',
  'resposta': 'O vale-presente Aurora vale por 12 meses e pode ser usado em qualquer produto do site.',
  'perguntas': ['Vocês têm vale-presente?',
                'Qual a validade do gift card?',
                'Posso dar um cartão-presente?',
                'Até quando posso usar o vale-presente?',
                'Como funciona o gift card de vocês?']},
 {'intencao': 'embalagem para presente',
  'chave': 'R$ 9,90',
  'resposta': 'A embalagem para presente custa R$ 9,90 e inclui um cartão com mensagem personalizada.',
  'perguntas': ['Vocês embrulham para presente?',
                'Quanto custa a embalagem de presente?',
                'Posso mandar com papel de presente?',
                'Tem opção de presente com cartão?',
                'Fazem pacote para presente?']},
 {'intencao': 'excluir conta',
  'chave': 'LGPD',
  'resposta': 'Você pode pedir a exclusão da conta e dos seus dados conforme a LGPD em Minha Conta > Privacidade.',
  'perguntas': ['Como excluo minha conta?',
                'Quero apagar meus dados.',
                'Como faço para vocês removerem minhas informações?',
                'Quero encerrar meu cadastro.',
                'Como deleto minha conta na loja?']},
 {'intencao': 'venda para empresas',
  'chave': 'CNPJ',
  'resposta': 'Para compras com CNPJ e acima de 10 unidades, use o Aurora Empresas, com preços e faturamento '
              'especiais.',
  'perguntas': ['Vocês vendem para empresas?',
                'Tem desconto para compra em quantidade?',
                'Posso comprar no CNPJ?',
                'Como faço compra corporativa?',
                'Quero comprar 50 unidades para minha empresa.']},
 {'intencao': 'avaliação de produto',
  'chave': '50 pontos',
  'resposta': 'Ao avaliar um produto comprado você ganha 50 pontos Aurora+ após a aprovação da avaliação.',
  'perguntas': ['Ganho algo por avaliar produto?',
                'Como escrevo uma review?',
                'Vale a pena deixar avaliação?',
                'Onde avalio o que comprei?',
                'Avaliar o produto dá pontos?']},
 {'intencao': 'pedido atrasado',
  'chave': 'cupom de R$ 20',
  'resposta': 'Se o pedido atrasar além do prazo informado, você recebe automaticamente um cupom de R$ 20 para a '
              'próxima compra.',
  'perguntas': ['Meu pedido está atrasado.',
                'Passou do prazo e não chegou.',
                'O que acontece se a entrega atrasar?',
                'Vocês compensam atraso?',
                'A entrega não chegou na data prevista.']}]

In [ ]:
import random
import re
import numpy as np
import pandas as pd

treino_pares = [(p, it["resposta"]) for it in FAQ for p in it["perguntas"][:-1]]
teste = pd.DataFrame([{"pergunta": it["perguntas"][-1], "chave": it["chave"], "intencao": i}
                      for i, it in enumerate(FAQ)])
if SMOKE:
    treino_pares, teste = treino_pares[:16], teste.head(4)
print(f"{len(treino_pares)} pares de treino | {len(teste)} perguntas de teste")
teste.head()

## 3. Baseline: LLM pré-treinado, sem ajuste
Vamos usar o [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct): um *decoder* (estilo GPT)
com 0,5 bilhão de parâmetros, licença Apache 2.0, que cabe com folga numa T4.

**Métrica:** a resposta contém o fato-chave da política (ex.: "45 dias")? É simples, objetiva e de negócio.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM)
tok.padding_side = "left"   # para gerar em lote
modelo = AutoModelForCausalLM.from_pretrained(LLM, dtype=torch.float32).to(DEVICE)

SISTEMA = ("Você é o assistente de atendimento da LojaAurora, um e-commerce brasileiro. "
           "Responda em português, em uma ou duas frases.")

def mensagens(pergunta, contexto=None):
    sistema = SISTEMA + (f"\nPolítica da loja relevante: {contexto}" if contexto else "")
    return [{"role": "system", "content": sistema}, {"role": "user", "content": pergunta}]

@torch.no_grad()
def responder(perguntas, contextos=None, lote=16):
    modelo.eval()
    contextos = contextos or [None] * len(perguntas)
    respostas = []
    for i in range(0, len(perguntas), lote):
        prompts = [tok.apply_chat_template(mensagens(p, c), tokenize=False, add_generation_prompt=True)
                   for p, c in zip(perguntas[i:i + lote], contextos[i:i + lote])]
        entrada = tok(prompts, return_tensors="pt", padding=True).to(DEVICE)
        saida = modelo.generate(**entrada, max_new_tokens=20 if SMOKE else 80, do_sample=False,
                                pad_token_id=tok.pad_token_id)
        respostas += tok.batch_decode(saida[:, entrada.input_ids.shape[1]:], skip_special_tokens=True)
    return [r.strip() for r in respostas]

normalizar = lambda s: re.sub(r"\s+", "", s.lower())

def taxa_acerto(respostas):
    return np.mean([normalizar(c) in normalizar(r) for r, c in zip(respostas, teste.chave)])

t0 = time.time()
teste["sem_ajuste"] = responder(teste.pergunta.tolist())
print(f"Acerto sem ajuste: {taxa_acerto(teste.sem_ajuste):.0%} ({time.time() - t0:.0f}s)")
pd.set_option("display.max_colwidth", 160)
teste[["pergunta", "chave", "sem_ajuste"]].head(6)

O modelo é fluente, mas **inventa** as políticas (alucinação). Ele não tem como saber que o prazo de troca da LojaAurora é 45 dias.

## 4. Fine-tuning com LoRA
Ajustar os 500M de pesos sairia caro. A **LoRA** (*Low-Rank Adaptation*) congela o modelo e treina só
pequenas matrizes $A \cdot B$ de posto baixo somadas às camadas de atenção e MLP. Em geral isso é menos de 2% dos parâmetros.

Detalhe importante: a perda só é calculada sobre a **resposta**, não sobre o prompt (mascaramos o prompt com `-100`).

In [ ]:
from peft import LoraConfig, get_peft_model

# TODO: crie um LoraConfig com r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM" e target_modules nas camadas lineares do Qwen (dica: print(modelo) mostra q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj)
...
modelo = get_peft_model(modelo, config)
modelo.print_trainable_parameters()

In [ ]:
def preparar(pergunta, resposta):
    prompt = tok.apply_chat_template(mensagens(pergunta), tokenize=False, add_generation_prompt=True)
    completo = tok.apply_chat_template(mensagens(pergunta) + [{"role": "assistant", "content": resposta}],
                                       tokenize=False)
    ids_prompt = tok(prompt, add_special_tokens=False).input_ids
    ids = tok(completo, add_special_tokens=False).input_ids
    rotulos = [-100] * len(ids_prompt) + ids[len(ids_prompt):]   # só a resposta conta na perda
    return ids, rotulos

def colar(exemplos):
    maior = max(len(ids) for ids, _ in exemplos)
    ids = [i + [tok.pad_token_id] * (maior - len(i)) for i, _ in exemplos]
    rotulos = [r + [-100] * (maior - len(r)) for _, r in exemplos]
    mascara = [[1] * len(i) + [0] * (maior - len(i)) for i, _ in exemplos]
    return {k: torch.tensor(v, device=DEVICE) for k, v in
            {"input_ids": ids, "labels": rotulos, "attention_mask": mascara}.items()}

exemplos = [preparar(p, r) for p, r in treino_pares]
EPOCAS, LOTE = (1, 8) if SMOKE else (4, 8)
otimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=2e-4)

modelo.train()
t0 = time.time()
for epoca in range(EPOCAS):
    random.Random(epoca).shuffle(exemplos)
    perdas = []
    for i in range(0, len(exemplos), LOTE):
        perda = modelo(**colar(exemplos[i:i + LOTE])).loss
        perda.backward()
        otimizador.step()
        otimizador.zero_grad()
        perdas.append(perda.item())
    print(f"época {epoca + 1}: perda média {np.mean(perdas):.3f} ({time.time() - t0:.0f}s)")

In [ ]:
teste["lora"] = responder(teste.pergunta.tolist())
print(f"Acerto sem ajuste: {taxa_acerto(teste.sem_ajuste):.0%} → com LoRA: {taxa_acerto(teste.lora):.0%}")
teste[["pergunta", "chave", "lora"]].head(6)

## 5. Alternativa sem treino: RAG (*Retrieval-Augmented Generation*)
1. Transformamos cada pergunta **já respondida** do FAQ num vetor (**embedding**) com um *encoder*
   ([multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small), da família BERT).
2. Para cada pergunta nova, buscamos a pergunta conhecida mais parecida (similaridade de cosseno) e pegamos a política dela.
3. Entregamos essa política ao LLM **original** (adaptador LoRA desligado) como contexto no prompt.

*Comparar pergunta com pergunta funciona melhor que pergunta com documento: no nosso FAQ, 84% × 68% de acerto na busca.*

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("intfloat/multilingual-e5-small", device=DEVICE)
embutir = lambda textos: embedder.encode(["query: " + t for t in textos], normalize_embeddings=True)  # o E5 pede o prefixo

conhecidas = [(p, i) for i, it in enumerate(FAQ) for p in it["perguntas"][:-1]]
E_base = embutir([p for p, _ in conhecidas])
intencao_base = np.array([i for _, i in conhecidas])

def buscar(perguntas):
    # TODO: ache, para cada pergunta, a pergunta conhecida mais similar (produto escalar = cosseno, pois os vetores estão normalizados) e devolva a intenção dela
    ...

mais_similar = buscar(teste.pergunta.tolist())
print(f"Busca acertou a política em {np.mean(mais_similar == teste.intencao):.0%} das perguntas")

with modelo.disable_adapter():   # LLM original, sem LoRA
    teste["rag"] = responder(teste.pergunta.tolist(), contextos=[FAQ[i]["resposta"] for i in mais_similar])
teste[["pergunta", "chave", "rag"]].head(6)

## 6. Comparação final

In [ ]:
resumo = pd.DataFrame({
    "Acerto do fato-chave": [taxa_acerto(teste[c]) for c in ("sem_ajuste", "lora", "rag")],
    "Precisa treinar?": ["não", "sim (minutos de GPU)", "não"],
    "Atualizar uma política": ["—", "retreinar", "editar o documento"],
    "Cita a fonte?": ["não", "não", "sim"],
}, index=["Só prompt", "Fine-tuning LoRA", "RAG"])
display(resumo.style.format({"Acerto do fato-chave": "{:.0%}"}))

In [ ]:
# TODO: faça uma pergunta nova (que não está no FAQ) aos três sistemas e compare
...

**Para discutir:**
- A política de troca mudou de 45 para 60 dias. Quanto custa atualizar cada sistema?
- Qual deles você colocaria em produção falando com cliente? Quais *guardrails* faltam?
- Na prática, os dois se combinam: **LoRA para tom e formato**, **RAG para fatos que mudam**.

### Extra: o mesmo com Gemini na Vertex AI
Num ambiente gerenciado o fluxo é igual, e o modelo fica do lado do Google Cloud. Exemplo com o SDK `google-genai`
(precisa de projeto GCP com a Vertex AI habilitada; confira o nome do modelo atual no console):
```python
from google import genai
cliente = genai.Client(vertexai=True, project="SEU_PROJETO", location="us-central1")
politica = FAQ[0]["resposta"]
r = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"{SISTEMA}\nPolítica: {politica}\nPergunta: Qual o prazo de troca?",
)
print(r.text)
```
A Vertex AI também oferece **fine-tuning supervisionado gerenciado** do Gemini: você envia um JSONL com pares
pergunta/resposta (como o nosso `treino_pares`) e paga por token de treino.